# Heat & Air Quality Hazard — facility level (v0.2)

Produces a facility-keyed heat and air quality hazard index for **all 357 California
carceral facilities**, replacing the tract-centroid joins of v0.1.

**Inputs**
- `data_sources/hazards/heat/loca2_facility_heat.csv` — LOCA2-CA daily extraction, one
  containing cell per facility, 14-model ensemble (model democracy), historic (1981–2010)
  and mid-century (2041–2070, ssp370). Relative- and absolute-threshold counts.
- `data_sources/hazards/heat/calenviroscreen50csv_d_12226.csv` — CalEnviroScreen 5.0 air
  quality percentiles, joined to facilities by census tract.

**Hazard equation** (per facility, per period, normalized across all 357 facilities):
1. daytime = 50/50 blend of `loca2_days_over_avg_plus10` (relative) and `loca2_days_over_90`
   (absolute), each divided by its cross-period max, then combined
2. night = `loca2_nights_over_p95` (relative), divided by its cross-period max
3. `temp = (day_blend + night_norm) / 2`
4. `H = temp × (1 + 0.3 × AQI_norm/100)` — AQI multiplicative modifier; missing AQI → ×1
5. scale 0–100 by cross-period max: `heat_hazard_idx = H / max(H) × 100`

Max-normalization (not min-max) means a facility with no exceedances scores 0.

**Output** `data/hazards/heat_air_hazard.csv` — 357 facilities, keyed on `facilityid`.
Carries the raw relative and absolute counts (so the CDCR index can renormalize across its 31
facilities), AQI, and the 357-normalized composite. The heat-risk index (CDCR-31) recomputes the
composite on its own 31-facility base; this file's composite is the all-facilities product.

In [1]:
import pandas as pd
import numpy as np

## 1. Load inputs

In [2]:
# LOCA2-CA facility heat product (357 facilities, one cell each)
loca = pd.read_csv('data_sources/hazards/heat/loca2_facility_heat.csv')
print(f'LOCA2 product: {len(loca)} facilities')

# tract_geoid per facility (the LOCA2 product is keyed by facilityid, not tract)
ca = pd.read_csv('data_sources/facilities/ca_facilities.csv')[['facilityid', 'tract_geoid']]
loca = loca.merge(ca, on='facilityid', how='left')
loca['tract_str'] = loca['tract_geoid'].astype(str).str.split('.').str[0].str.zfill(11)
print(f'tract_geoid nulls: {loca["tract_geoid"].isna().sum()}')

LOCA2 product: 357 facilities
tract_geoid nulls: 0


In [3]:
# AQI from CalEnviroScreen 5.0: mean of ozone, PM2.5, diesel percentiles, then
# min-max normalized 0-100 across all CA tracts (identical definition to v0.1).
# encoding='utf-8-sig' strips the BOM from the first column name.
ces = pd.read_csv('data_sources/hazards/heat/calenviroscreen50csv_d_12226.csv',
                  encoding='utf-8-sig', dtype={'tract': str})
ces['tract'] = ces['tract'].str.zfill(11)
ces['AQI'] = ces[['ozoneP', 'pmP', 'dieselP']].mean(axis=1)
aqi_min, aqi_max = ces['AQI'].min(), ces['AQI'].max()
ces['AQI_norm'] = (ces['AQI'] - aqi_min) / (aqi_max - aqi_min) * 100
print(f'CalEnviroScreen tracts: {len(ces)}; AQI_norm range {ces["AQI_norm"].min():.1f}-{ces["AQI_norm"].max():.1f}')

loca = loca.merge(ces[['tract', 'AQI_norm']], left_on='tract_str', right_on='tract', how='left')
print(f'AQI_norm nulls after join: {loca["AQI_norm"].isna().sum()} of {len(loca)} (treated as x1 in the modifier)')
cdcr = loca[loca['cdcr_code'].notna()]
print(f'CDCR facilities AQI_norm range: {cdcr["AQI_norm"].min():.1f}-{cdcr["AQI_norm"].max():.1f}')

CalEnviroScreen tracts: 9106; AQI_norm range 0.0-100.0
AQI_norm nulls after join: 0 of 357 (treated as x1 in the modifier)
CDCR facilities AQI_norm range: 12.2-94.4


## 2. Composite hazard index

Daytime heat is a 50/50 blend of a facility-relative threshold (days above the facility's mean
summer daily-max + 10°F, 1981–2010 baseline) and an absolute threshold (days over 90°F), each
max-normalized before blending. Warm nights are April–October nights above the P95 of the
facility's 1961–1990 April–October tmin (OEHHA convention). The blended daytime term and the
warm-night term are averaged, then multiplied by the air-quality modifier. The 50% blend weight
is a parameter (`W_REL`); after normalization the 80°F-vs-90°F choice is immaterial (ρ≈0.95).

In [4]:
def build_hazard(df, rel_day_cols, abs_day_cols, night_cols,
                 aqi_col='AQI_norm', beta=0.30, w_rel=0.50):
    """v0.2 hazard equation, normalized across the rows of `df` and both periods (0-100).

    Daytime heat is a blend of a facility-relative threshold (days over the facility's own
    summer mean + 10 F) and an absolute threshold (days over 90 F), each max-normalized before
    blending, with weight w_rel on the relative term. Nights use the P95 relative threshold.
    AQI enters as a multiplicative modifier."""
    rd_h, rd_m = rel_day_cols
    ad_h, ad_m = abs_day_cols
    n_h, n_m = night_cols

    rd_max = df[[rd_h, rd_m]].to_numpy().max()
    ad_max = df[[ad_h, ad_m]].to_numpy().max()
    n_max  = df[[n_h, n_m]].to_numpy().max()

    def day(rel, ab):
        return w_rel * (df[rel] / rd_max) + (1 - w_rel) * (df[ab] / ad_max)
    day_h, day_m = day(rd_h, ad_h), day(rd_m, ad_m)
    night_h, night_m = df[n_h] / n_max, df[n_m] / n_max

    temp_h = (day_h + night_h) / 2
    temp_m = (day_m + night_m) / 2

    # AQI multiplicative modifier; missing AQI -> x1
    modifier = 1 + beta * (df[aqi_col].fillna(0) / 100)
    H_h = temp_h * modifier
    H_m = temp_m * modifier

    H_max = pd.concat([H_h, H_m]).max()
    return pd.DataFrame({
        'heat_hazard_historic_idx':   H_h / H_max * 100,
        'heat_hazard_midcentury_idx': H_m / H_max * 100,
    }, index=df.index)


W_REL = 0.50  # daytime relative/absolute blend weight (0.50 = equal); a documented design parameter
haz = build_hazard(
    loca,
    rel_day_cols=('loca2_days_over_avg_plus10_historic', 'loca2_days_over_avg_plus10_midcentury'),
    abs_day_cols=('loca2_days_over_90_historic', 'loca2_days_over_90_midcentury'),
    night_cols=('loca2_nights_over_p95_historic', 'loca2_nights_over_p95_midcentury'),
    w_rel=W_REL,
)
loca = loca.join(haz)
print(f'Composite heat hazard (357 facilities, 0-100), daytime {W_REL:.0%} relative / {1-W_REL:.0%} absolute-90F:')
print(loca[['heat_hazard_historic_idx', 'heat_hazard_midcentury_idx']].describe().round(2))

Composite heat hazard (357 facilities, 0-100), daytime 50% relative / 50% absolute-90F:
       heat_hazard_historic_idx  heat_hazard_midcentury_idx
count                    357.00                      357.00
mean                      27.74                       69.54
std                        5.78                       13.58
min                       14.21                       34.13
25%                       23.27                       61.95
50%                       27.06                       71.91
75%                       32.20                       78.97
max                       43.13                      100.00


## 3. Export

In [5]:
output = loca[[
    # keys
    'facilityid', 'name', 'cdcr_code', 'tract_geoid',
    # relative-threshold counts (index inputs; renormalized across CDCR-31 downstream)
    'loca2_days_over_avg_plus10_historic', 'loca2_days_over_avg_plus10_midcentury',
    'loca2_nights_over_p95_historic', 'loca2_nights_over_p95_midcentury',
    # absolute daytime count, retained for display continuity with v0.1
    'loca2_days_over_90_historic', 'loca2_days_over_90_midcentury',
    # relative-threshold baselines (auditability)
    'loca2_avg_summer_tmax_f', 'loca2_p95_tmin_f',
    # air quality
    'AQI_norm',
    # 357-normalized composite (all-facilities product)
    'heat_hazard_historic_idx', 'heat_hazard_midcentury_idx',
]].copy()

output.to_csv('data/hazards/heat_air_hazard.csv', index=False)
print(f'Wrote {len(output)} facilities x {len(output.columns)} columns -> data/hazards/heat_air_hazard.csv')
print('\nTop 10 CDCR facilities by mid-century composite:')
print(output[output['cdcr_code'].notna()]
      .sort_values('heat_hazard_midcentury_idx', ascending=False)
      [['cdcr_code', 'loca2_days_over_avg_plus10_midcentury', 'loca2_nights_over_p95_midcentury',
        'AQI_norm', 'heat_hazard_midcentury_idx']].head(10).round(2).to_string(index=False))

Wrote 357 facilities x 15 columns -> data/hazards/heat_air_hazard.csv

Top 10 CDCR facilities by mid-century composite:
cdcr_code  loca2_days_over_avg_plus10_midcentury  loca2_nights_over_p95_midcentury  AQI_norm  heat_hazard_midcentury_idx
      CIM                                  36.91                             46.87     94.40                       93.05
      CIW                                  37.01                             46.62     75.59                       89.27
      CRC                                  33.50                             47.94     59.58                       86.50
      COR                                  19.45                             51.85     61.69                       84.13
     SATF                                  19.45                             51.85     61.69                       84.13
     CCWF                                  23.70                             50.05     54.67                       81.92
      VSP                        